Overview 
- Extract comprehensive structured signals from each listing: amenities (pool, fireplace), condition (updated, new), financing (seller financing), location features. Combine Week 3 entities with taxonomy-based pattern matching. 

Key Deliverables 
- SignalExtractor class processing full listing records ✅
- JSON output schema: entities + amenities + keywords ✅
- Process entire rets_property table (save to processed/) ✅
- Extraction accuracy: 90%+ for structured fields, 75%+ for free text ➖ (achieved 85% for structured fields and 75%+ for free text) - if time permits, work on implementing ner spaCy model
- Output suitable for search indexing and filtering ✅

Deliverable 1: Creating the SignalExtractor class and process the full listing records

In [1]:
import os
import sys
import json
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import spacy

In [2]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.w3_entity_extractor import EntityExtractor

In [3]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from scripts.w2_text_cleaning import TextCleaner

========== REMARKS PROFILE ==========

Total listings: 1000
Null rate: 0.00%
Average length: 1290.09 characters

Price mentions: 57
Measurement mentions: 261
Room dimensions: 7

HTML tags: 0
Unicode usages: 394
Smart quotes: 276
Whitespace issues: 482
APN Counts: 3
Phone Number Counts: 0

Known abbreviations
ft       251
sq       237
condo    139
hoa      134
adu      122
bbq      104
rv       70
ev       58
hvac     57
ac       45

Unknown abbreviations
altos    3
sony     3
ages     3
wings    3
pulls    3
hemet    3
harte    3
rents    3
sj       3
tools    3
finds    3
items    3
lofts    3
gpm      2
draws    2
verde    2
lies     2
rises    2
tells    2
baja     2

Top 20 words
home         2383
living       1964
room         1233
offers       1073
space        1068
kitchen      953
private      849
bedroom      832
s            824
dining       815
2            796
features     761
you          750
area         690
located      668
spacious     667
bedrooms     666
new          

In [4]:
with open("../data/processed/taxonomy.json", "r") as f:
    taxonomy = json.load(f)

In [5]:
print(type(taxonomy))
print(list(taxonomy.items())[:2])

<class 'dict'>
[('categories', ['kitchen', 'flooring', 'outdoor', 'parking', 'housing_layout', 'housing_features', 'location', 'community_amenities']), ('terms', [{'id': 1, 'term': 'primary suite', 'category': 'housing_layout', 'frequency': 336}, {'id': 2, 'term': 'natural light', 'category': 'housing_features', 'frequency': 294}, {'id': 3, 'term': 'living room', 'category': 'housing_layout', 'frequency': 283}, {'id': 4, 'term': 'living space', 'category': 'housing_layout', 'frequency': 263}, {'id': 5, 'term': 'floor plan', 'category': 'housing_layout', 'frequency': 261}, {'id': 6, 'term': 'stainless steel', 'category': 'kitchen', 'frequency': 193}, {'id': 8, 'term': 'stainless steel appliances', 'category': 'kitchen', 'frequency': 172}, {'id': 9, 'term': 'shopping dining', 'category': 'location', 'frequency': 171}, {'id': 10, 'term': 'located near', 'category': 'location', 'frequency': 159}, {'id': 11, 'term': 'conveniently located', 'category': 'location', 'frequency': 158}, {'id': 1

In [6]:
df = pd.read_csv('../data/processed/cleaned_listing_sample.csv')
df.shape

(1000, 8)

In [7]:
import re

class SignalExtractor: 
    def __init__(self, taxonomy, entity_extractor): 
        self.taxonomy = taxonomy 
        self.extractor = entity_extractor 
        
    def extract_signals(self, listing_record): 
        remarks = listing_record.get('cleaned_remarks', '') 

        # Get entities from Week 3 
        entities = self.extractor.extract_all(remarks) 

        # Get amenities from taxonomy 
        amenities = self._match_amenities(remarks) 

        # Detect signals 
        return { 
        'listing_id': listing_record['L_ListingID'], 
        'entities': entities, 
        'amenities': amenities, 
        'condition_keywords': self._extract_condition(remarks), 
        'financing_terms': self._extract_financing(remarks), 
        'location_features': self._extract_location(remarks) 
        }

    def _match_amenities(self, remarks):
        matches = []
        remarks = remarks.lower()

        for item in self.taxonomy['terms']:
            term = item['term'].lower()

            if term in remarks:
                matches.append({
                    "term": item["term"],
                    "category": item["category"]
                })

        return matches

    def _extract_condition(self, remarks):

        if not isinstance(remarks, str):
            return []

        condition_patterns = {
            "updated": [
                r"\bupdated\b",
                r"\bupgraded\b",
                r"\bmodernized\b",
            ],

            "renovated": [
                r"\brenovat(?:ed|ion|ions|ing)\b",
            ],

            "remodeled": [
                r"\bremodel(?:ed|ing)?\b",
                r"\bremodeled\b",
                r"\bfully remodeled\b",
                r"\bpartially remodeled\b",
            ],

            "new construction": [
                r"\bnew construction\b",
                r"\bnewly constructed\b",
                r"\bnew construction home\b",
            ],

            "newly built": [
                r"\bnewly built\b",
                r"\bnew build\b",
                r"\bnewly constructed\b",
            ],

            "needs renovation": [
                r"\bneeds renovation\b",
                r"\bneeds renovations\b",
                r"\bneeds updating\b",
                r"\bneeds repair\b",
                r"\bneeds repairs\b",
                r"\bneeds work\b",
                r"\bfixer[- ]upper\b",
                r"\bfixer\b",
            ],

            "move-in ready": [
                r"\bmove[- ]in ready\b",
                r"\bmove[- ]in condition\b",
            ],

            "well-maintained": [
                r"\bwell[- ]maintained\b",
                r"\bwell[- ]kept\b",
                r"\bmeticulously maintained\b",
                r"\bpride of ownership\b",
            ],

            "recently updated": [
                r"\brecently updated\b",
                r"\brecently upgraded\b",
            ],

            "recently renovated": [
                r"\brecently renovated\b",
            ],

            "recently remodeled": [
                r"\brecently remodeled\b",
            ],

            "brand new": [
                r"\bbrand new\b",
            ],
        }

        remarks = remarks.lower()

        matches = []

        for label, patterns in condition_patterns.items():

            for pattern in patterns:

                if re.search(pattern, remarks):
                    matches.append(label)
                    break

        return matches

    def _extract_financing(self, remarks):

        if not isinstance(remarks, str):
            return []

        financing_patterns = {

            "seller financing": [
                r"\bseller financing\b",
                r"\bseller[- ]financed\b",
                r"\bseller[- ]financing\b",
                r"\bseller carry\b",
                r"\bseller carryback\b",
                r"\bseller will finance\b",
            ],

            "owner financing": [
                r"\bowner financing\b",
                r"\bowner[- ]financed\b",
                r"\bowner[- ]financing\b",
                r"\bowner carry\b",
                r"\bowner carryback\b",
                r"\bowner will finance\b",
            ],

            "assumable loan": [
                r"\bassumable loan\b",
                r"\bassumable mortgage\b",
                r"\bloan assumption\b",
                r"\bmortgage assumption\b",
                r"\bassume the loan\b",
                r"\bassume existing loan\b",
            ],

            "financing available": [
                r"\bfinancing available\b",
                r"\bfinancing options?\b",
                r"\bfinancing option\b",
                r"\bfinancing terms\b",
                r"\bfinancing offered\b",
            ],

            "cash only": [
                r"\bcash only\b",
                r"\bcash buyers? only\b",
                r"\bcash purchase\b",
                r"\bcash offers? only\b",
            ],

            "cash offer": [
                r"\bcash offer\b",
                r"\bcash offers\b",
            ],

            "lease option": [
                r"\blease option\b",
                r"\blease[- ]to[- ]own\b",
                r"\brent[- ]to[- ]own\b",
                r"\blease purchase\b",
                r"\blease[- ]purchase\b",
            ],

            "loan approved": [
                r"\bloan approved\b",
                r"\bapproved loan\b",
                r"\bpre[- ]approved\b",
                r"\bpreapproved\b",
            ],
        }

        remarks = remarks.lower()

        matches = []

        for label, patterns in financing_patterns.items():

            for pattern in patterns:

                if re.search(pattern, remarks):
                    matches.append(label)
                    break

        return matches

    def _extract_location(self, remarks):

        if not isinstance(remarks, str):
            return []

        location_patterns = {

            "near schools": [
                r"\bnear schools?\b",
                r"\bclose to schools?\b",
                r"\bclose to local schools?\b",
            ],

            "near shopping": [
                r"\bnear shopping\b",
                r"\bclose to shopping\b",
                r"\bnear shopping centers?\b",
                r"\bclose to shopping centers?\b",
            ],

            "near restaurants": [
                r"\bnear restaurants?\b",
                r"\bclose to restaurants?\b",
                r"\bnear dining\b",
                r"\bclose to dining\b",
            ],

            "near downtown": [
                r"\bnear downtown\b",
                r"\bclose to downtown\b",
                r"\bminutes from downtown\b",
                r"\bnearby downtown\b",
            ],

            "near parks": [
                r"\bnear parks?\b",
                r"\bclose to parks?\b",
                r"\bnearby parks?\b",
            ],

            "freeway access": [
                r"\bnear freeways?\b",
                r"\bclose to freeways?\b",
                r"\beasy freeway access\b",
                r"\beasy access to (?:the )?freeways?\b",
                r"\bfreeway access\b",
                r"\bnear major highways?\b",
                r"\bclose to major highways?\b",
            ],

            "public transportation": [
                r"\bnear public transportation\b",
                r"\bnear public transit\b",
                r"\bnear transit\b",
                r"\bpublic transportation\b",
            ],

            "walking distance": [
                r"\bwalking distance\b",
                r"\bwithin walking distance\b",
                r"\bwalkable\b",
            ],

            "near beach": [
                r"\bnear beaches?\b",
                r"\bclose to beaches?\b",
                r"\bnear the beach\b",
                r"\bclose to the beach\b",
            ],

            "mountain views": [
                r"\bmountain views?\b",
                r"\bviews of the mountains\b",
                r"\bmountain view\b",
            ],

            "hill views": [
                r"\bhill views?\b",
                r"\bhillside views?\b",
            ],

            "city views": [
                r"\bcity views?\b",
                r"\bviews of the city\b",
            ],

            "ocean views": [
                r"\bocean views?\b",
                r"\bocean view\b",
                r"\bviews of the ocean\b",
            ],

            "water views": [
                r"\bwater views?\b",
                r"\bwater view\b",
            ],

            "lake views": [
                r"\blake views?\b",
                r"\blake view\b",
            ],

            "valley views": [
                r"\bvalley views?\b",
                r"\bvalley view\b",
            ],

            "scenic views": [
                r"\bscenic views?\b",
                r"\bscenic view\b",
                r"\bpanoramic views?\b",
                r"\bpanoramic view\b",
            ],

            "golf course": [
                r"\bgolf course\b",
                r"\bgolf course views?\b",
                r"\bgolf course view\b",
            ],

            "private beach access": [
                r"\bprivate beach access\b",
            ],
        }

        remarks = remarks.lower()

        matches = []

        for label, patterns in location_patterns.items():

            for pattern in patterns:

                if re.search(pattern, remarks):
                    matches.append(label)
                    break

        return matches

In [12]:
extractor = EntityExtractor(taxonomy)
signal_extractor = SignalExtractor(taxonomy, extractor)

In [13]:
print(len(signal_extractor.taxonomy["terms"]))
print(signal_extractor.taxonomy["terms"][:3])

138
[{'id': 1, 'term': 'primary suite', 'category': 'housing_layout', 'frequency': 336}, {'id': 2, 'term': 'natural light', 'category': 'housing_features', 'frequency': 294}, {'id': 3, 'term': 'living room', 'category': 'housing_layout', 'frequency': 283}]


In [11]:
matched_count = 0

for _, row in df.iterrows():
    remarks = str(row["cleaned_remarks"]).lower()

    for item in taxonomy["terms"]:
        if item["term"].lower() in remarks:
            matched_count += 1
            break

print("Listings with at least one taxonomy match:", matched_count)

Listings with at least one taxonomy match: 962


962 out of 1000 listings have at least one taxonomy match

In [12]:
condition_terms  = [
            "updated",
            "renovated",
            "newly renovated",
            "newly updated",
            "newly built",
            "new construction",
            "needs renovation",
            "needs updating",
            "needs repair",
            "needs work",
            "fixer-upper",
            "fixer",
            "move-in ready",
            "well-maintained",
            "well-kept",
            "recently remodeled",
            "recently updated",
            "recently renovated",
            "recently built",
            "recently constructed",
            "brand new"
        ]

In [13]:
condition_counts = []

for _, row in df.iterrows():
    remarks = str(row["cleaned_remarks"]).lower()

    matches = [
        term for term in condition_terms
        if term in remarks
    ]

    if matches:
        condition_counts.append({
            "listing_id": row["L_ListingID"],
            "matches": matches
        })

print("Listings with condition signals:", len(condition_counts))

Listings with condition signals: 417


417 out of 1000 listings have at least one condition term

In [14]:
financing_terms = [
            "seller financing",
            "seller-financed",
            "seller-financing",
            "owner financing",
            "owner-financed",
            "owner-financing",
            "owner carry",
            "owner carryback",
            "seller carry",
            "seller carryback",
            "assumable loan",
            "assumable mortgage",
            "financing available",
            "financing option",
            "cash only",
            "cash offer",
            "lease option",
            "rent to own",
            "rent-to-own",
            "lease purchase",
            "lease-purchase",
            "loan approved",
            "loan assumption"
      ]

In [15]:
finance_counts = []

for _, row in df.iterrows():
    remarks = str(row["cleaned_remarks"]).lower()

    matches = [
        term for term in financing_terms
        if term in remarks
    ]

    if matches:
        finance_counts.append({
            "listing_id": row["L_ListingID"],
            "matches": matches
        })

print("Listings with financing signals:", len(finance_counts))

Listings with financing signals: 12


In [16]:
location_terms = [
        "near schools",
        "near school",
        "close to schools",
        "close to school",
        "near shopping",
        "close to shopping",
        "near restaurants",
        "close to restaurants",
        "near downtown",
        "close to downtown",
        "near parks",
        "close to parks",
        "near freeway",
        "near freeways",
        "close to freeway",
        "close to freeways",
        "easy freeway access",
        "easy access to freeway",
        "near public transportation",
        "near transit",
        "walking distance",
        "walkable",
        "minutes from downtown",
        "minutes from shopping",
        "minutes from schools",
        "near beach",
        "near beaches",
        "close to beach",
        "close to beaches",
        "near major highways",
        "close to major highways",
        "mountain views",
        "mountain view",
        "mountain",
        "hill views",
        "hillside",
        "hillside views",
        "city views",
        "city view",
        "ocean views",
        "ocean view",
        "water views",
        "water view",
        "lake views",
        "lake view",
        "valley views",
        "valley view",
        "scenic views",
        "scenic view",
        "river views",
        "river view",
        "golf course views",
        "golf course view",
        "golf course",
        "panoramic views",
        "panormatic view",
        "private beach access",
        ]

In [17]:
location_counts = []

for _, row in df.iterrows():
    remarks = str(row["cleaned_remarks"]).lower()

    matches = [
        term for term in location_terms
        if term in remarks
    ]

    if matches:
        location_counts.append({
            "listing_id": row["L_ListingID"],
            "matches": matches
        })

print("Listings with location signals:", len(location_counts))

Listings with location signals: 467


In [19]:
listing = row.to_dict()

signals = signal_extractor.extract_signals(listing)

signals

{'listing_id': 1152252457,
 'entities': {'bedrooms': None,
  'bathrooms': '>1',
  'price': None,
  'sqft': None,
  'amenities': [{'term': 'primary suite', 'category': 'housing_layout'},
   {'term': 'stainless steel', 'category': 'kitchen'},
   {'term': 'stainless steel appliances', 'category': 'kitchen'},
   {'term': 'easy access', 'category': 'location'},
   {'term': 'laundry room', 'category': 'housing_features'},
   {'term': 'private balcony', 'category': 'outdoor'},
   {'term': 'flooring throughout', 'category': 'flooring'},
   {'term': 'tile flooring', 'category': 'flooring'},
   {'term': 'gated community', 'category': 'community_amenities'},
   {'term': 'fitness center', 'category': 'community_amenities'},
   {'term': 'remodeled kitchen', 'category': 'kitchen'},
   {'term': 'prime location', 'category': 'location'},
   {'term': 'tennis courts', 'category': 'community_amenities'},
   {'term': 'parking space', 'category': 'parking'}]},
 'amenities': [{'term': 'primary suite', 'cate

Deliverable 3 - Process entire rets_property table (save to processed/) - but first do 1000 sample

In [20]:
os.makedirs("../data/processed", exist_ok=True)

In [21]:
all_signals = []

for _, row in df.iterrows():
    listing = row.to_dict()

    signals = signal_extractor.extract_signals(listing)

    all_signals.append(signals)

print("Listings processed:", len(all_signals))

Listings processed: 1000


In [22]:
all_signals[5] # example

{'listing_id': 1158077880,
 'entities': {'bedrooms': 3,
  'bathrooms': 2,
  'price': None,
  'sqft': 1560,
  'amenities': [{'term': 'living space', 'category': 'housing_layout'},
   {'term': 'easy access', 'category': 'location'}]},
 'amenities': [{'term': 'living space', 'category': 'housing_layout'},
  {'term': 'easy access', 'category': 'location'}],
 'condition_keywords': ['fixer'],
 'financing_terms': [],
 'location_features': ['close to shopping']}

In [23]:
output_path = "../data/processed/listing_signals.json"
with open(output_path, "w", encoding = "utf-8") as f:
    json.dump(all_signals, f, indent=2, default=str)
print(f"Saved {len(all_signals)} listings to {output_path}")

Saved 1000 listings to ../data/processed/listing_signals.json


Now the entire retsproperty table

In [24]:
load_dotenv()
def get_connection():
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST"),
            user=os.getenv("MYSQL_USER"),
            password=os.getenv("MYSQL_PASSWORD"),
            database=os.getenv("MYSQL_DATABASE")
        )

        return conn

In [25]:
conn = get_connection()
cursor = conn.cursor(dictionary=True)
query = """ 
SELECT L_ListingID, L_Address, L_City, L_Keyword2 as beds, 
LM_Dec_3 as baths, L_SystemPrice as price, LM_Int2_3 as sqft, L_Remarks as remarks 
FROM rets_property 
WHERE L_Remarks IS NOT NULL 
""" 
df_full = pd.read_sql(query, conn)

print("Listings loaded:", len(df_full))

C:\Users\mayab\AppData\Local\Temp\ipykernel_25332\3333261372.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_full = pd.read_sql(query, conn)


Listings loaded: 52794


52794 out of the 53122 listings have non-null remarks

In [26]:
df_full.head()

,L_ListingID,L_Address,L_City,beds,baths,price,sqft,remarks
0,1118422731,1461 Laurel Way,Beverly Hills,4.0,5.0,3950000,3677.0,Opportunity to reimagine a Classic 70's Archit...
1,1118417283,13545 Surcease Mine Road,Oroville,3.0,1.0,199000,950.0,Gated Privacy with views of Lake Oroville. Th...
2,1118405579,31499 Via Las Rosas,Carmel Valley,4.0,3.0,2890000,2826.0,Your search ends here. This spectacular prope...
3,1118398412,582 Pasture Ave,Lathrop,4.0,3.0,775000,2709.0,Step into comfort and functionality with this ...
4,1119757884,13679 Calle Romano,Tijuana,3.0,4.0,480000,2637.0,"Introducing Helena Residences, an exclusive de..."


In [27]:
df_full.isna().sum()

L_ListingID      0
L_Address      114
L_City          92
beds           101
baths           17
price            0
sqft            83
remarks          0
dtype: int64

In [ ]:
'''cleaner = TextCleaner()

df_full["cleaned_remarks"] = (
    df_full["remarks"]
    .apply(cleaner.clean_text)
)'''

'cleaner = TextCleaner()\n\ndf_full["cleaned_remarks"] = (\n    df_full["remarks"]\n    .apply(cleaner.clean_text)\n)'

In [ ]:
#df_full.to_csv("../data/processed/cleaned_listing_full.csv", index=False, encoding="utf-8")

In [8]:
df_full = pd.read_csv("../data/processed/cleaned_listing_full.csv", encoding="utf-8")

In [29]:
df_full.shape

(52794, 9)

In [14]:
problem_listing = df_full[
    df_full["L_ListingID"] == 1130466836
].iloc[0].to_dict()

signals = signal_extractor.extract_signals(problem_listing)

print(signals)

{'listing_id': 1130466836, 'entities': {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': None}, 'amenities': [{'term': 'private patio', 'category': 'outdoor'}], 'condition_keywords': [], 'financing_terms': [], 'location_features': ['public transportation']}


In [ ]:
'''all_signals = []

for _, row in df_full.iterrows():
    listing = row.to_dict()

    signals = signal_extractor.extract_signals(listing)

    all_signals.append(signals)

print("Listings processed:", len(all_signals))'''

'all_signals = []\n\nfor _, row in df_full.iterrows():\n    listing = row.to_dict()\n\n    signals = signal_extractor.extract_signals(listing)\n\n    all_signals.append(signals)\n\nprint("Listings processed:", len(all_signals))'

In [ ]:
'''output_path = "../data/processed/listing_signals_full.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_signals, f, indent=2)

print(f"Saved {len(all_signals)} listings to {output_path}")'''

'output_path = "../data/processed/listing_signals_full.json"\nwith open(output_path, "w", encoding="utf-8") as f:\n    json.dump(all_signals, f, indent=2)\n\nprint(f"Saved {len(all_signals)} listings to {output_path}")'

Deliverable 2 - Defining and demonstrating the JSON output schema: 

SignalExtractor output

        ↓

JSON schema/example

        ↓

example_signal_output.json

Done above

In [31]:
with open(output_path, "r", encoding="utf-8") as f:
    loaded_signals = json.load(f)

print("Listings loaded:", len(loaded_signals))
print(loaded_signals[0])

Listings loaded: 1000
{'listing_id': 1169503734, 'entities': {'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}, 'amenities': [], 'condition_keywords': [], 'financing_terms': [], 'location_features': []}


In [32]:
print(all_signals[0])

{'listing_id': 1169503734, 'entities': {'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}, 'amenities': [], 'condition_keywords': [], 'financing_terms': [], 'location_features': []}


Deliverable 4 - Extraction Accuracy: Asking to prove that SignalExtractor is extracting useful information accurately

- Structured fields -> target: 90%+ - extracting bedrooms, bathrooms, price, and square footage
- Free-text signals -> target: 75%+ - looking through cleaned_remarks to extract amenities, condition keywords, financing terms, and location features

Since we have the entire rets_property table at hand, we will take a sample of the listings and run the SignalExtractor on them. Then we will compare the actual bedrooms to extracted bedrooms, actual bathrooms to the extracted, and then expected sqft to its extracted. Next we will calculate the accuracy for each and then calculate the overall structured-field accuracy to determine whether we meet the 90% target with the full table.

In [33]:
df_full.columns.tolist()

['L_ListingID',
 'L_Address',
 'L_City',
 'beds',
 'baths',
 'price',
 'sqft',
 'remarks',
 'cleaned_remarks']

In [34]:
evaluation_sample = df_full.copy()

In [35]:
evaluation_sample["extracted_bedrooms"] = evaluation_sample["cleaned_remarks"].apply(
    extractor.extract_bedrooms
)

evaluation_sample["extracted_bathrooms"] = evaluation_sample["cleaned_remarks"].apply(
    extractor.extract_bathrooms
)

evaluation_sample["extracted_price"] = evaluation_sample["cleaned_remarks"].apply(
    extractor.extract_price
)

evaluation_sample["extracted_sqft"] = evaluation_sample["cleaned_remarks"].apply(
    extractor.extract_sqft
)

In [36]:
evaluation_sample.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'sqft',
       'remarks', 'cleaned_remarks', 'extracted_bedrooms',
       'extracted_bathrooms', 'extracted_price', 'extracted_sqft'],
      dtype='object')

In [37]:
fields = {
    "bedrooms": ("beds", "extracted_bedrooms"),
    "bathrooms": ("baths", "extracted_bathrooms"),
    "price": ("price", "extracted_price"),
    "sqft": ("sqft", "extracted_sqft")
}

accuracies = {}

In [38]:
def bed_bath_match(true, pred):
    """
    Match bedrooms/bathrooms.

    A prediction of '>1' is considered correct when
    the actual value is greater than 1.
    """

    # Both are missing
    if pd.isna(true) and pd.isna(pred):
        return True

    # Actual missing, prediction exists
    if pd.isna(true) and not pd.isna(pred):
        return False

    # Actual exists, prediction missing
    if not pd.isna(true) and pd.isna(pred):
        return False

    # '>1' is a valid prediction for any actual value > 1
    if str(pred).strip() == ">1":
        return true > 1

    # Otherwise require an exact match
    return true == pred


def price_match(
    true,
    pred,
    percent_tolerance=0.20,
    absolute_tolerance=50000
):
    """
    Match extracted price against the actual price.

    Rules:
    - '>X' is correct if actual price is greater than X.
    - '<X' is correct if actual price is less than X.
    - Numeric predictions are correct if they are either:
        - within 20% of the actual price, OR
        - within $50,000 of the actual price.
    """

    # Both are missing
    if pd.isna(true) and pd.isna(pred):
        return True

    # Actual missing, prediction exists
    if pd.isna(true) and not pd.isna(pred):
        return False

    # Actual exists, prediction missing
    if not pd.isna(true) and pd.isna(pred):
        return False

    # Convert actual price to number
    true = float(true)

    # Convert prediction to string
    pred_str = str(pred).strip()

    # --------------------------------------------------
    # Handle greater-than predictions
    # --------------------------------------------------

    if pred_str.startswith(">"):
        try:
            threshold = float(
                pred_str[1:].replace(",", "").strip()
            )
        except ValueError:
            return False

        return true > threshold

    # --------------------------------------------------
    # Handle less-than predictions
    # --------------------------------------------------

    if pred_str.startswith("<"):
        try:
            threshold = float(
                pred_str[1:].replace(",", "").strip()
            )
        except ValueError:
            return False

        return true < threshold

    # --------------------------------------------------
    # Handle normal numeric prediction
    # --------------------------------------------------

    try:
        pred_value = float(
            pred_str.replace(",", "").replace("$", "")
        )
    except ValueError:
        return False

    # Calculate absolute difference
    difference = abs(true - pred_value)

    # Calculate relative difference
    relative_difference = difference / true

    # Accept if either tolerance is satisfied
    return (
        difference <= absolute_tolerance
        or relative_difference <= percent_tolerance
    )


def sqft_match(true, pred, tolerance=0.05):
    """
    Match extracted sqft against the actual sqft.

    A prediction is considered correct if it is within
    5% of the actual square footage.
    """

    # Missing values are not correct
    if pd.isna(true) or pd.isna(pred):
        return False

    # Convert to numbers
    true = float(true)
    pred = float(pred)

    # Prevent division by zero
    if true == 0:
        return pred == 0

    # Calculate relative difference
    relative_difference = abs(true - pred) / true

    # Accept values within 5%
    return relative_difference <= tolerance


for name, (actual_col, extracted_col) in fields.items():

    # Start with rows where the actual database value exists
    evaluation = evaluation_sample[
        evaluation_sample[actual_col].notna()
    ].copy()

    # --------------------------------------------------
    # Only evaluate when REMARKS contain this field
    # --------------------------------------------------

    if name == "bedrooms":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d+\s*(?:bed(?:room)?s?|br)\b",
            case=False,
            na=False
        )

    elif name == "bathrooms":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d+(?:\.\d+)?\s*(?:bath(?:room)?s?|ba)\b",
            case=False,
            na=False
        )

    elif name == "price":

        # Match:
        #   >100000
        #   <2000000
        #   $1,500,000
        #   1,500,000
        mask = evaluation["cleaned_remarks"].str.contains(
            r'(?:'
            r'[<>]\s*\$?\s*\d[\d,]*'
            r'|'
            r'\$\s*\d[\d,]*'
            r'|'
            r'\b\d{3,}(?:,\d{3})+\b'
            r')',
            case=False,
            na=False
        )

    elif name == "sqft":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d[\d,]*\s*(?:sq\.?\s*ft\.?|square\s+feet|sf)\b",
            case=False,
            na=False
        )

    # Keep ONLY listings where remarks mention the field
    evaluation = evaluation[mask]

    # Remove cases where extractor did not return a value
    evaluation = evaluation[
        evaluation[extracted_col].notna()
    ]

    # --------------------------------------------------
    # Calculate correctness
    # --------------------------------------------------

    if name in ["bedrooms", "bathrooms"]:

        # Use Week 3 logic, including valid '>1'
        correct = evaluation.apply(
            lambda row: bed_bath_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    elif name == "price":

        # Use relational matching for >X / <X
        # and tolerance matching for numeric prices
        correct = evaluation.apply(
            lambda row: price_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    elif name == "sqft":

        # Use 5% tolerance for sqft
        correct = evaluation.apply(
            lambda row: sqft_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    # --------------------------------------------------
    # Calculate accuracy
    # --------------------------------------------------

    accuracy = (
        correct.mean() * 100
        if len(evaluation) > 0
        else 0
    )

    accuracies[name] = accuracy

    print(
        f"{name.capitalize()} accuracy: "
        f"{accuracy:.2f}% "
        f"({len(evaluation)} evaluated)"
    )

Bedrooms accuracy: 86.24% (16216 evaluated)
Bathrooms accuracy: 84.49% (15153 evaluated)
Price accuracy: 85.13% (948 evaluated)
Sqft accuracy: 85.24% (13927 evaluated)


In [39]:
total_correct = 0
total_evaluated = 0

for name, (actual_col, extracted_col) in fields.items():

    # Start with rows where actual value exists
    evaluation = evaluation_sample[
        evaluation_sample[actual_col].notna()
    ].copy()

    # --------------------------------------------------
    # Only evaluate when REMARKS contain this field
    # --------------------------------------------------

    if name == "bedrooms":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d+\s*(?:bed(?:room)?s?|br)\b",
            case=False,
            na=False
        )

    elif name == "bathrooms":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d+(?:\.\d+)?\s*(?:bath(?:room)?s?|ba)\b",
            case=False,
            na=False
        )

    elif name == "price":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\$\s*\d[\d,]*|\b\d{3,}(?:,\d{3})+\b",
            case=False,
            na=False
        )

    elif name == "sqft":

        mask = evaluation["cleaned_remarks"].str.contains(
            r"\b\d[\d,]*\s*(?:sq\.?\s*ft\.?|square\s+feet|sf)\b",
            case=False,
            na=False
        )

    # Keep only listings where remarks mention the field
    evaluation = evaluation[mask]

    # Remove cases where extractor returned nothing
    evaluation = evaluation[
        evaluation[extracted_col].notna()
    ]

    # --------------------------------------------------
    # Apply the correct matching rule
    # --------------------------------------------------

    if name in ["bedrooms", "bathrooms"]:

        correct = evaluation.apply(
            lambda row: bed_bath_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    elif name == "price":

        correct = evaluation.apply(
            lambda row: price_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    elif name == "sqft":

        correct = evaluation.apply(
            lambda row: sqft_match(
                row[actual_col],
                row[extracted_col]
            ),
            axis=1
        )

    # Add this field's results to overall totals
    total_correct += correct.sum()
    total_evaluated += len(correct)


# --------------------------------------------------
# Overall accuracy
# --------------------------------------------------

overall_accuracy = (
    (total_correct / total_evaluated) * 100
    if total_evaluated > 0
    else 0
)

print(f"Overall extraction accuracy: {overall_accuracy:.2f}")
print(f"Total field values evaluated: {total_evaluated}")
print(f"Total correct: {total_correct}")

Overall extraction accuracy: 85.34
Total field values evaluated: 46244
Total correct: 39466


The structured extraction pipeline currently achieves approximately 85% field-level accuracy. The remaining errors primarily occur when MLS remarks contain multiple historical, auction, parcel, or non-property dollar values. Further improvements would require more contextual disambiguation rather than simple pattern matching.

For the free-text, this would include amenities, condition, financing, and location features - then would evaluate each category against the labeled sample and calculate the free-text accuracy. 

In [21]:
evaluation_sample = df.sample(
    n=200,
    random_state=42
).copy()

In [22]:
evaluation_sample["signals"] = evaluation_sample.apply(
    lambda row: signal_extractor.extract_signals(row.to_dict()),
    axis=1
)

In [24]:
evaluation_sample[["L_ListingID", "signals"]].head()

,L_ListingID,signals
521,1174195371,"{'listing_id': 1174195371, 'entities': {'bedro..."
737,1154290282,"{'listing_id': 1154290282, 'entities': {'bedro..."
740,1157630609,"{'listing_id': 1157630609, 'entities': {'bedro..."
660,1157498637,"{'listing_id': 1157498637, 'entities': {'bedro..."
411,1173668830,"{'listing_id': 1173668830, 'entities': {'bedro..."


In [25]:
evaluation_sample["extracted_amenities"] = evaluation_sample["signals"].apply(
    lambda x: [item["term"] for item in x["amenities"]]
)

evaluation_sample["extracted_condition"] = evaluation_sample["signals"].apply(
    lambda x: x["condition_keywords"]
)

evaluation_sample["extracted_financing"] = evaluation_sample["signals"].apply(
    lambda x: x["financing_terms"]
)

evaluation_sample["extracted_location"] = evaluation_sample["signals"].apply(
    lambda x: x["location_features"]
)

In [27]:
# Generate the taxonomy candidates
def find_expected_terms(text, taxonomy): 
    if not isinstance(text, str): 
        return [] 
 
    found_terms = [] 
 
    for term in taxonomy["terms"]:
        term_text = term.get("term")
 
        if not term_text:
            continue
 
        if term_text.lower() in text.lower():
            found_terms.append(term_text)
 
    return found_terms

In [28]:
evaluation_sample["taxonomy_matches"] = evaluation_sample[
    "cleaned_remarks"
].apply(
    lambda text: find_expected_terms(text, taxonomy)
)

In [29]:
def get_taxonomy_terms_by_category(text, taxonomy, category):
    if not isinstance(text, str):
        return []

    text_lower = text.lower()
    matches = []

    for item in taxonomy["terms"]:
        term = item.get("term")
        
        if not term:
            continue

        if item.get("category") != category:
            continue

        if term.lower() in text_lower:
            matches.append(term)

    return matches

In [31]:
AMENITY_CATEGORIES = {
    "kitchen",
    "flooring",
    "outdoor",
    "parking",
    "housing_layout",
    "housing_features",
    "community_amenities"
}

In [32]:
def get_taxonomy_amenities(text, taxonomy):
    if not isinstance(text, str):
        return []

    text_lower = text.lower()
    matches = []

    for item in taxonomy["terms"]:
        term = item.get("term")
        category = item.get("category")

        if not term or category not in AMENITY_CATEGORIES:
            continue

        if term.lower() in text_lower:
            matches.append(term)

    return matches

In [33]:
evaluation_sample["taxonomy_amenities"] = evaluation_sample[
    "cleaned_remarks"
].apply(
    lambda text: get_taxonomy_amenities(text, taxonomy)
)

In [48]:
def calculate_match_score(extracted, expected):
    extracted = set(extracted)
    expected = set(expected)

    # Nothing expected and nothing extracted = correct
    if not expected and not extracted:
        return 1.0

    # Expected terms exist, but extractor found nothing
    if expected and not extracted:
        return 0.0

    # Extractor found terms when none were expected
    if extracted and not expected:
        return 0.0

    # Both contain terms: calculate proportion correctly matched
    overlap = extracted & expected

    return len(overlap) / len(expected)

In [49]:
evaluation_sample["amenity_match_rate"] = evaluation_sample.apply(
    lambda row: calculate_match_score(
        row["extracted_amenities"],
        row["taxonomy_amenities"]
    ),
    axis=1
)

In [50]:
evaluation_sample["amenity_match_rate"].mean()

0.985

In [51]:
CONDITION_TERMS = [
    "updated",
    "renovated",
    "newly renovated",
    "newly updated",
    "newly built",
    "new construction",
    "needs renovation",
    "needs updating",
    "needs repair",
    "needs work",
    "fixer-upper",
    "fixer",
    "move-in ready",
    "well-maintained",
    "well-kept",
    "recently remodeled",
    "recently updated",
    "recently renovated",
    "recently built",
    "recently constructed",
    "brand new"
]

In [52]:
def find_terms_in_text(text, terms):
    if not isinstance(text, str):
        return []

    text_lower = text.lower()

    return [
        term for term in terms
        if term.lower() in text_lower
    ]

In [53]:
evaluation_sample["condition_text_matches"] = evaluation_sample[
    "cleaned_remarks"
].apply(
    lambda text: find_terms_in_text(text, CONDITION_TERMS)
)

In [54]:
evaluation_sample["condition_match_rate"] = evaluation_sample.apply(
    lambda row: calculate_match_score(
        row["extracted_condition"],
        row["condition_text_matches"]
    ),
    axis=1
)

In [55]:
FINANCING_TERMS = [
    "seller financing",
    "seller-financed",
    "seller-financing",
    "owner financing",
    "owner-financed",
    "owner-financing",
    "owner carry",
    "owner carryback",
    "seller carry",
    "seller carryback",
    "assumable loan",
    "assumable mortgage",
    "financing available",
    "financing option",
    "cash only",
    "cash offer",
    "lease option",
    "rent to own",
    "rent-to-own",
    "lease purchase",
    "lease-purchase",
    "loan approved",
    "loan assumption"
]

In [56]:
evaluation_sample["financing_text_matches"] = evaluation_sample[
    "cleaned_remarks"
].apply(
    lambda text: find_terms_in_text(text, FINANCING_TERMS)
)

In [57]:
evaluation_sample["financing_match_rate"] = evaluation_sample.apply(
    lambda row: calculate_match_score(
        row["extracted_financing"],
        row["financing_text_matches"]
    ),
    axis=1
)

In [58]:
LOCATION_TERMS = [
        "near schools",
        "near school",
        "close to schools",
        "close to school",
        "near shopping",
        "close to shopping",
        "near restaurants",
        "close to restaurants",
        "near downtown",
        "close to downtown",
        "near parks",
        "close to parks",
        "near freeway",
        "near freeways",
        "close to freeway",
        "close to freeways",
        "easy freeway access",
        "easy access to freeway",
        "near public transportation",
        "near transit",
        "walking distance",
        "walkable",
        "minutes from downtown",
        "minutes from shopping",
        "minutes from schools",
        "near beach",
        "near beaches",
        "close to beach",
        "close to beaches",
        "near major highways",
        "close to major highways",
        "mountain views",
        "mountain view",
        "mountain",
        "hill views",
        "hillside",
        "hillside views",
        "city views",
        "city view",
        "ocean views",
        "ocean view",
        "water views",
        "water view",
        "lake views",
        "lake view",
        "valley views",
        "valley view",
        "scenic views",
        "scenic view",
        "river views",
        "river view",
        "golf course views",
        "golf course view",
        "golf course",
        "panoramic views",
        "panormatic view",
        "private beach access",
        ]

In [59]:
evaluation_sample["location_text_matches"] = evaluation_sample[
    "cleaned_remarks"
].apply(
    lambda text: find_terms_in_text(text, LOCATION_TERMS)
)

In [60]:
evaluation_sample["location_match_rate"] = evaluation_sample.apply(
    lambda row: calculate_match_score(
        row["extracted_location"],
        row["location_text_matches"]
    ),
    axis=1
)

In [61]:
summary = {
    "Amenities": evaluation_sample["amenity_match_rate"].mean(),
    "Condition": evaluation_sample["condition_match_rate"].mean(),
    "Financing": evaluation_sample["financing_match_rate"].mean(),
    "Location": evaluation_sample["location_match_rate"].mean()
}

summary

{'Amenities': 0.985,
 'Condition': 0.7983333333333335,
 'Financing': 0.995,
 'Location': 0.6889166666666668}

In [69]:
print(f"Overall accuracy: {((evaluation_sample['amenity_match_rate'].mean() + evaluation_sample['condition_match_rate'].mean() + evaluation_sample['financing_match_rate'].mean() + evaluation_sample['location_match_rate'].mean()) / 4).round(2)}")

Overall accuracy: 0.87
